# Book Recommender — Phase 1: Data Collection

## Overview

This notebook collects 2,000+ books from **4 sources** using 2 techniques:

| # | Technique | Source | Target |
|---|-----------|--------|--------|
| 1 | Web Scraping | books.toscrape.com | ~999 books |
| 2 | Web Scraping | books.toscrape.com (by genre detail pages) | ~200 extra |
| 3 | API | Open Library (30 subjects × 100 books) | ~1,500 books |
| 4 | API | Gutendex — Project Gutenberg classics | ~300 books |

**Minimum target: 2,000 unique books**

**Output:** `raw_books.csv`

## 1. Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
import json

print('All libraries imported successfully')
print('pandas:', pd.__version__)

## 2. Web Scraping — books.toscrape.com

### How web scraping works
```
1. REQUEST  → Python sends HTTP GET to the website URL
2. RECEIVE  → Website returns raw HTML text
3. PARSE    → BeautifulSoup reads the HTML tree structure
4. EXTRACT  → We find specific tags and pull out the data
5. PAGINATE → Follow the next button through all pages
```

### HTML structure we target
```html
<article class='product_pod'>      <- one book per article
  <p class='star-rating Three'>   <- rating stored as CSS class word
  <h3><a title='Full Title'>      <- full title in title attribute
  <p class='price_color'>£51.77  <- price
</article>
```

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
SCRAPE_BASE_URL = 'http://books.toscrape.com'
DELAY = 0.5
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}
print('Scraping configuration ready')
print(f'Target: {SCRAPE_BASE_URL}')

In [ ]:
# ── Scraping helper functions ───────────────────────────────────────────────

def get_page(url):
    """Send HTTP GET request and return parsed BeautifulSoup object."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'lxml')
    except Exception as e:
        print(f'  Request failed: {e}')
        return None


def parse_books_from_page(soup, genre):
    """Extract all book records from one list page."""
    books = []
    for article in soup.find_all('article', class_='product_pod'):
        # Title — stored in the 'title' attribute (full version, not truncated)
        h3    = article.find('h3')
        a_tag = h3.find('a') if h3 else None
        title = a_tag['title'] if a_tag else 'Unknown'

        # Rating — second CSS class is the word e.g. 'Three'
        star_tag = article.find('p', class_='star-rating')
        rating   = RATING_MAP.get(star_tag['class'][1], 0) if star_tag else 0

        # Price — remove pound sign
        price_tag = article.find('p', class_='price_color')
        price_txt = price_tag.get_text(strip=True) if price_tag else '0'
        price     = float(re.sub(r'[^\d.]', '', price_txt)) if price_txt else None

        books.append({
            'title':          title,
            'genre':          genre,
            'scraped_rating': rating,
            'price':          price,
            'source':         'WebScraping'
        })
    return books


def get_all_genre_urls(soup):
    """Extract every genre link from the sidebar navigation."""
    genres = {}
    nav = soup.find('ul', class_='nav-list')
    if nav:
        for link in nav.find_all('a')[1:]:  # skip first 'Books' link
            name = link.get_text(strip=True)
            url  = f"{SCRAPE_BASE_URL}/{link['href']}"
            genres[name] = url
    return genres


def scrape_full_genre(genre_name, genre_url):
    """Scrape ALL pages of one genre — follows pagination until end."""
    all_books = []
    url       = genre_url

    while True:
        soup = get_page(url)
        if not soup:
            break
        books = parse_books_from_page(soup, genre_name)
        all_books.extend(books)

        # Check for next page button
        next_btn = soup.find('li', class_='next')
        if next_btn:
            next_href = next_btn.find('a')['href']
            base      = genre_url.rsplit('/', 1)[0]
            url       = f"{base}/{next_href}"
            time.sleep(DELAY)
        else:
            break

    return all_books


print('Scraping functions defined successfully')

In [ ]:
# ── Run web scraper — all 50 genres ────────────────────────────────────────
print('Starting web scraping from books.toscrape.com...')
print('=' * 55)

homepage = get_page(SCRAPE_BASE_URL)
genres   = get_all_genre_urls(homepage)
print(f'Found {len(genres)} genres to scrape\n')

scraped_books = []
for i, (genre_name, genre_url) in enumerate(genres.items(), 1):
    books = scrape_full_genre(genre_name, genre_url)
    scraped_books.extend(books)
    print(f'[{i:2d}/{len(genres)}] {genre_name:<25} {len(books):>3} books')
    time.sleep(DELAY)

scraped_df = pd.DataFrame(scraped_books)
print(f'\nWeb scraping complete: {len(scraped_df):,} books collected')
scraped_df.head(3)

## 3. Open Library API — 30 subjects × 100 books

### What is an API?
An API returns **structured JSON data** — cleaner and more reliable than HTML parsing.

### Key difference from scraping
```python
# Scraping:  response.text -> BeautifulSoup -> find HTML tags
# API:       response.json() -> Python dict -> access by key name
```

### Endpoint
```
https://openlibrary.org/search.json
    ?subject=fantasy     <- search by genre
    &limit=100           <- 100 results per call
    &sort=rating         <- highest rated first
    &fields=title,...    <- only return fields we need
```

In [ ]:
# ── Open Library configuration — 30 subjects × 100 books = ~3,000 requests
OL_BASE_URL = 'https://openlibrary.org/search.json'

OL_SUBJECTS = [
    # Core fiction genres
    'fiction', 'fantasy', 'mystery', 'romance', 'science_fiction',
    'thriller', 'horror', 'adventure', 'historical_fiction', 'classics',
    # Non-fiction
    'biography', 'history', 'science', 'psychology', 'philosophy',
    'politics', 'self_help', 'travel', 'cooking', 'nature',
    # Other genres
    'poetry', 'children', 'young_adult', 'graphic_novels', 'memoir',
    'true_crime', 'art', 'music', 'sports', 'short_stories'
]

BOOKS_PER_SUBJECT = 100  # 30 x 100 = 3,000 requests -> ~1,500 unique after dedup

print(f'Open Library: {len(OL_SUBJECTS)} subjects x {BOOKS_PER_SUBJECT} books')
print(f'Expected: ~{len(OL_SUBJECTS) * BOOKS_PER_SUBJECT} requests -> ~1,500 unique books')

In [ ]:
# ── Test a single API call first ────────────────────────────────────────────
print('Testing Open Library API with a single request...')

test_params = {
    'subject': 'fantasy',
    'limit':   3,
    'fields':  'title,author_name,first_publish_year,subject,ratings_average,ratings_count'
}
test_response = requests.get(OL_BASE_URL, params=test_params, timeout=10)
print(f'Status code:      {test_response.status_code}')

test_data = test_response.json()
print(f'Total available:  {test_data.get("numFound", 0):,} books')
print(f'Returned:         {len(test_data.get("docs", []))} books')
print(f'\nFirst book raw JSON:')
print(json.dumps(test_data['docs'][0], indent=2))

In [ ]:
# ── Open Library fetch function ─────────────────────────────────────────────

def fetch_open_library(subject, limit=100):
    """
    Fetch books from Open Library Search API.

    The API returns a JSON object with a 'docs' list.
    Each doc is a dictionary — we extract fields using .get() with defaults.
    .get('field', default) returns default if field is missing.
    """
    params = {
        'subject': subject,
        'limit':   limit,
        'fields':  (
            'key,title,author_name,first_publish_year,'
            'number_of_pages_median,ratings_average,'
            'ratings_count,subject,isbn'
        ),
        'sort': 'rating'
    }
    try:
        response = requests.get(OL_BASE_URL, params=params, timeout=10)
        response.raise_for_status()
        docs = response.json().get('docs', [])
    except Exception as e:
        print(f'  API error ({subject}): {e}')
        return []

    books = []
    for doc in docs:
        books.append({
            'title':         doc.get('title', 'Unknown'),
            'authors':       ', '.join(doc.get('author_name', ['Unknown'])),
            'genre':         subject.replace('_', ' ').title(),
            'subjects':      '; '.join(doc.get('subject', [])[:6]),
            'avg_rating':    doc.get('ratings_average', None),
            'ratings_count': doc.get('ratings_count', 0),
            'year':          doc.get('first_publish_year', None),
            'pages':         doc.get('number_of_pages_median', None),
            'isbn':          doc.get('isbn', [None])[0] if doc.get('isbn') else None,
            'source':        'OpenLibraryAPI'
        })
    return books


print('Open Library function defined')

In [ ]:
# ── Run Open Library collection ─────────────────────────────────────────────
print('Collecting from Open Library API...')
print(f'30 subjects x 100 books — estimated time: 4-6 minutes')
print('=' * 55)

ol_books = []
for i, subject in enumerate(OL_SUBJECTS, 1):
    books = fetch_open_library(subject, limit=BOOKS_PER_SUBJECT)
    ol_books.extend(books)
    print(f'[{i:2d}/{len(OL_SUBJECTS)}] {subject:<25} {len(books):>3} books  (running total: {len(ol_books):,})')
    time.sleep(DELAY)

ol_df = pd.DataFrame(ol_books)
print(f'\nOpen Library complete: {len(ol_df):,} books')
ol_df.head(3)

## 4. Gutendex API — Project Gutenberg Classics

Gutendex is a free API for Project Gutenberg — over 70,000 classic books. No API key required.

### Endpoint
```
https://gutendex.com/books/?page=1
```
Returns 32 books per page. We collect 15 pages = ~480 classic books.

### Why classics?
Books like Pride and Prejudice, Moby Dick, Sherlock Holmes — well known titles that improve recommendation quality.

In [ ]:
# ── Test Gutendex API ───────────────────────────────────────────────────────
GUTENDEX_URL = 'https://gutendex.com/books/'

print('Testing Gutendex API...')
test_g = requests.get(GUTENDEX_URL, params={'page': 1}, timeout=10)
print(f'Status: {test_g.status_code}')

if test_g.status_code == 200:
    gdata = test_g.json()
    print(f'Total books available: {gdata.get("count", 0):,}')
    print(f'Books per page: {len(gdata.get("results", []))}')
    print(f'\nFirst book:')
    first = gdata['results'][0]
    print(f'  Title:   {first["title"]}')
    print(f'  Authors: {[a["name"] for a in first["authors"]]}')
    print(f'  Subjects:{first["subjects"][:3]}')
else:
    print('Gutendex not reachable — will skip this source')

In [ ]:
# ── Gutendex fetch function ─────────────────────────────────────────────────

def fetch_gutendex(page=1):
    """
    Fetch classic books from Gutendex (Project Gutenberg API).
    Returns 32 books per page.
    Authors are stored as a list of dicts: [{'name': 'Austen, Jane'}]
    """
    try:
        r = requests.get(GUTENDEX_URL, params={'page': page}, timeout=10)
        r.raise_for_status()
        results = r.json().get('results', [])
    except Exception as e:
        print(f'  Error page {page}: {e}')
        return []

    books = []
    for book in results:
        # Authors stored as list of dicts
        authors   = ', '.join([a['name'] for a in book.get('authors', [])])
        # Subjects as a list — use first as genre
        subjects  = book.get('subjects', [])
        genre     = subjects[0].split('--')[0].strip()[:50] if subjects else 'Classic'
        # Bookshelves as additional subject tags
        shelves   = book.get('bookshelves', [])

        books.append({
            'title':         book.get('title', 'Unknown')[:200],
            'authors':       authors or 'Unknown',
            'genre':         genre,
            'subjects':      '; '.join(subjects[:5] + shelves[:3]),
            'avg_rating':    None,
            'ratings_count': book.get('download_count', 0),
            'year':          None,
            'pages':         None,
            'isbn':          None,
            'source':        'GutendexAPI'
        })
    return books


print('Gutendex function defined')

In [ ]:
# ── Run Gutendex — 15 pages x 32 books = ~480 classics ─────────────────────
print('Collecting from Gutendex API...')
print('=' * 55)

gutendex_books = []
GUTENDEX_PAGES = 15

for page in range(1, GUTENDEX_PAGES + 1):
    books = fetch_gutendex(page)
    gutendex_books.extend(books)
    print(f'  Page {page:2d}/{GUTENDEX_PAGES}: {len(books)} books  (running total: {len(gutendex_books):,})')
    time.sleep(DELAY)

gutendex_df = pd.DataFrame(gutendex_books) if gutendex_books else pd.DataFrame()
print(f'\nGutendex complete: {len(gutendex_df):,} books')
if not gutendex_df.empty:
    gutendex_df.head(3)

## 5. Combine All Sources

In [ ]:
# ── Standardise columns across all sources ──────────────────────────────────

COLS = ['title', 'authors', 'genre', 'subjects', 'avg_rating',
        'ratings_count', 'year', 'pages', 'isbn', 'source']

# Scraped — add missing columns
scraped_std = scraped_df.copy()
scraped_std['authors']       = 'Unknown'
scraped_std['subjects']      = scraped_std['genre']
scraped_std['avg_rating']    = np.nan
scraped_std['ratings_count'] = np.nan
scraped_std['year']          = np.nan
scraped_std['pages']         = np.nan
scraped_std['isbn']          = np.nan
scraped_std = scraped_std.reindex(columns=COLS)

# Open Library — already has all columns
ol_std = ol_df.reindex(columns=COLS)

# Gutendex — already has all columns
gutendex_std = gutendex_df.reindex(columns=COLS) if not gutendex_df.empty else pd.DataFrame(columns=COLS)

print(f'Scraped:      {len(scraped_std):,} books')
print(f'Open Library: {len(ol_std):,} books')
print(f'Gutendex:     {len(gutendex_std):,} books')
print(f'Raw total:    {len(scraped_std)+len(ol_std)+len(gutendex_std):,} books')

In [ ]:
# ── Concatenate and deduplicate ──────────────────────────────────────────────

frames = [scraped_std, ol_std]
if not gutendex_std.empty:
    frames.append(gutendex_std)

combined = pd.concat(frames, ignore_index=True)

print(f'Before deduplication: {len(combined):,} books')
combined = combined.drop_duplicates(subset=['title'])
combined = combined.reset_index(drop=True)
print(f'After deduplication:  {len(combined):,} books')
print(f'\nSource breakdown:')
print(combined['source'].value_counts())

In [ ]:
# ── Clean combined dataset ───────────────────────────────────────────────────

# Convert numeric columns
combined['avg_rating']    = pd.to_numeric(combined['avg_rating'], errors='coerce')
combined['ratings_count'] = pd.to_numeric(combined['ratings_count'], errors='coerce')
combined['year']          = pd.to_numeric(combined['year'], errors='coerce')
combined['pages']         = pd.to_numeric(combined['pages'], errors='coerce')

# Fill missing text fields
combined['authors']  = combined['authors'].fillna('Unknown')
combined['subjects'] = combined['subjects'].fillna(combined['genre'])

# Remove empty titles
combined = combined[
    combined['title'].notna() &
    (combined['title'].str.strip() != '')
].reset_index(drop=True)

# Create content field for TF-IDF in Phase 3
# This combines all text signals into one field
combined['content'] = (
    combined['title'].fillna('') + ' ' +
    combined['authors'].fillna('') + ' ' +
    combined['genre'].fillna('') + ' ' +
    combined['subjects'].fillna('')
)

print(f'Final cleaned dataset: {len(combined):,} books')
print(f'\nMissing values:')
print(combined.isnull().sum())

## 6. Dataset Summary & Charts

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.facecolor']  = 'white'
plt.rcParams['axes.facecolor']    = '#FAFAFA'

BLUE  = '#2C3E50'
GREEN = '#27AE60'
RED   = '#E74C3C'
AMBER = '#F39C12'
PURPLE= '#8E44AD'
PALETTE = [BLUE, GREEN, AMBER, PURPLE, RED]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1 — Books by source
src    = combined['source'].value_counts()
colors = PALETTE[:len(src)]
bars   = axes[0].bar(src.index, src.values, color=colors, edgecolor='white')
for bar, val in zip(bars, src.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5, f'{val:,}',
                 ha='center', fontweight='600', fontsize=10)
axes[0].set_title('Books by Collection Source', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Number of Books')
axes[0].tick_params(axis='x', rotation=20)

# Chart 2 — Top genres
excl   = ['mixed', 'default', 'add a comment']
genres = combined[~combined['genre'].str.lower().isin(excl)]['genre'].value_counts().head(12)
axes[1].barh(genres.index[::-1], genres.values[::-1], color=BLUE, edgecolor='white')
axes[1].set_title('Top 12 Genres', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Number of Books')

# Chart 3 — Rating distribution (API books only)
rated = combined[combined['avg_rating'].notna()]['avg_rating']
if len(rated) > 0:
    axes[2].hist(rated, bins=25, color=GREEN, edgecolor='white', alpha=0.85)
    axes[2].axvline(rated.mean(), color=RED, linestyle='--',
                    linewidth=2, label=f'Mean: {rated.mean():.2f}')
    axes[2].axvline(rated.median(), color=AMBER, linestyle='--',
                    linewidth=2, label=f'Median: {rated.median():.2f}')
    axes[2].set_title('Rating Distribution (Open Library)', fontweight='bold', fontsize=12)
    axes[2].set_xlabel('Average Rating')
    axes[2].legend()

plt.suptitle('Data Collection Summary', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_data_collection.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary printout
print('FINAL DATASET SUMMARY')
print('=' * 40)
print(f'Total books:          {len(combined):,}')
print(f'Web scraped:          {(combined["source"]=="WebScraping").sum():,}')
print(f'Open Library API:     {(combined["source"]=="OpenLibraryAPI").sum():,}')
print(f'Gutendex API:         {(combined["source"]=="GutendexAPI").sum():,}')
print(f'Unique authors:       {combined["authors"].nunique():,}')
print(f'Genres covered:       {combined["genre"].nunique()}')
print(f'Books with ratings:   {combined["avg_rating"].notna().sum():,}')
print(f'Average rating:       {combined["avg_rating"].mean():.2f}')
print(f'Target met (2000+):   {"YES" if len(combined) >= 2000 else "NO — need more sources"}')

In [ ]:
# ── Save final dataset ───────────────────────────────────────────────────────
combined.to_csv('raw_books.csv', index=False)
print(f'Saved raw_books.csv: {len(combined):,} books')
print(f'Columns: {combined.columns.tolist()}')
print('\nNext step: open and run eda.ipynb')

## 7. Technique Comparison

| Feature | Web Scraping | Open Library API | Gutendex API |
|---------|-------------|-----------------|-------------|
| Method | Parse HTML | Request JSON | Request JSON |
| Auth needed | No | No | No |
| Real ratings | No (random) | Yes | No |
| Author data | No | Yes | Yes |
| Book type | Modern | Mixed | Classics only |
| Fragility | High | Low | Low |
| Books collected | ~999 | ~1,500 unique | ~480 |

**Key insight:** No single source is perfect. Combining multiple sources gives broader coverage, richer metadata, and a more representative dataset — which is standard practice in real data science projects.